EDA Workshop

Task 1

In [ ]:
import requests
import pandas as pd
import sqlite3

In [ ]:
import requests

#Add your own API_Key

url = "https://api.themoviedb.org/3/movie/changes"

headers = {
    "accept": "application/json",
    "Authorization": "API_Key-------"
}

response = requests.get(url, headers=headers)

print(response.text)

{"results":[{"id":1657501,"adult":false},{"id":14,"adult":false},{"id":15534,"adult":false},{"id":862,"adult":false},{"id":17532,"adult":false},{"id":1657503,"adult":false},{"id":250045,"adult":false},{"id":1657321,"adult":false},{"id":1575466,"adult":false},{"id":1648624,"adult":true},{"id":1375987,"adult":false},{"id":1479319,"adult":false},{"id":1507189,"adult":false},{"id":1650393,"adult":false},{"id":1657162,"adult":false},{"id":1098342,"adult":false},{"id":1246032,"adult":false},{"id":1359767,"adult":false},{"id":1441336,"adult":false},{"id":1504896,"adult":false},{"id":1563886,"adult":false},{"id":1591909,"adult":false},{"id":1620148,"adult":false},{"id":1624916,"adult":false},{"id":1631668,"adult":false},{"id":1636489,"adult":false},{"id":1637483,"adult":false},{"id":1640299,"adult":false},{"id":1641689,"adult":false},{"id":1641816,"adult":false},{"id":1645797,"adult":false},{"id":1645802,"adult":false},{"id":1645805,"adult":false},{"id":1645807,"adult":false},{"id":1645810,"ad

In [ ]:
API_KEY = "5fce4e09faa4a56cb97bae0400c49695"

url = "https://api.themoviedb.org/3/discover/movie"

params = {
    "api_key": API_KEY,
    "language": "en-US",
    "sort_by": "popularity.desc",
    "page": 1  # 1 page = 20 movies
}

response = requests.get(url, params=params)
data = response.json()

In [ ]:
movies = data['results']

df = pd.DataFrame(movies)

# Select useful columns only (avoid unnecessary noise)
df = df[[
    'id', 'title', 'release_date', 'popularity',
    'vote_average', 'vote_count', 'genre_ids', 'overview'
]]

print(df.head())

        id                             title release_date  popularity  \
0   875828  Peaky Blinders: The Immortal Man   2026-03-05    360.2727   
1   687163                 Project Hail Mary   2026-03-15    319.2442   
2    83533              Avatar: Fire and Ash   2025-12-17    318.2150   
3  1290821                           Shelter   2026-01-28    305.7725   
4  1265609                       War Machine   2026-02-12    284.7653   

   vote_average  vote_count      genre_ids  \
0         7.400         421       [80, 18]   
1         8.182         494      [878, 12]   
2         7.267        1930  [878, 12, 14]   
3         6.700         385   [28, 80, 53]   
4         7.278        1150  [28, 878, 53]   

                                            overview  
0  After his estranged son gets embroiled in a Na...  
1  Science teacher Ryland Grace wakes up on a spa...  
2  In the wake of the devastating war against the...  
3  A man living in self-imposed exile on a remote...  
4  On one

Task 2 — Perform EDA

In [ ]:
import ast

conn = sqlite3.connect("movies.db")

# Convert the 'genre_ids' column from list to string representation before saving
df['genre_ids'] = df['genre_ids'].apply(str)

# Save the DataFrame to the SQLite database
# if_exists='replace' will overwrite the table if it already exists
# index=False prevents writing the DataFrame index as a column
df.to_sql('movies', conn, if_exists='replace', index=False)

# Now, read the data from the newly created table
df = pd.read_sql("SELECT * FROM movies", conn)

# Convert the 'genre_ids' column back from string to list after reading
df['genre_ids'] = df['genre_ids'].apply(ast.literal_eval)

conn.close()

In [ ]:
print(df.head())

Empty DataFrame
Columns: [id, title, release_date, popularity, vote_average, vote_count, genre_ids, overview]
Index: []


In [ ]:
print(df.describe())

                 id  popularity  vote_average   vote_count
count  2.000000e+01   20.000000     20.000000    20.000000
mean   1.117265e+06  206.523485      6.895350   578.850000
std    3.470208e+05   89.085988      0.776937   613.575481
min    8.353300e+04  110.804900      5.007000     3.000000
25%    1.032097e+06  119.310700      6.394250   204.500000
50%    1.196248e+06  183.621250      7.015000   440.500000
75%    1.312296e+06  294.074500      7.450000   708.500000
max    1.634301e+06  360.272700      8.182000  2328.000000


In [ ]:
# Convert string representation to list (if needed)
import ast
df['genre_ids'] = df['genre_ids'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Explode genre list
df_exploded = df.explode('genre_ids')

genre_counts = df_exploded['genre_ids'].value_counts()

print(genre_counts)

genre_ids
53       7
12       6
878      6
28       6
18       5
27       5
80       4
35       4
9648     4
16       3
14       2
10749    2
10751    2
Name: count, dtype: int64


In [ ]:
genre_url = "https://api.themoviedb.org/3/genre/movie/list"

genre_response = requests.get(genre_url, params={"api_key": API_KEY})
genre_data = genre_response.json()

genre_map = {g['id']: g['name'] for g in genre_data['genres']}

df_exploded['genre_name'] = df_exploded['genre_ids'].map(genre_map)

print(df_exploded['genre_name'].value_counts())

genre_name
Thriller           7
Adventure          6
Science Fiction    6
Action             6
Drama              5
Horror             5
Crime              4
Comedy             4
Mystery            4
Animation          3
Fantasy            2
Romance            2
Family             2
Name: count, dtype: int64


In [ ]:
print(df.isnull().sum())

id              0
title           0
release_date    0
popularity      0
vote_average    0
vote_count      0
genre_ids       0
overview        0
dtype: int64
